In [8]:
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import faiss
import time
from deepeval import evaluate
import json

In [9]:
model = SentenceTransformer("BAAI/bge-small-en-v1.5")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 5192.44it/s]


In [12]:
with open("500DaysofSummer.txt", "r", encoding="utf-8") as file:
    text = file.read()

In [14]:
def chunk_text(text, chunk_size=400, overlap=100):
    chunks = []
    for i in range(0, len(text), chunk_size - overlap):
        chunk = text[i:i + chunk_size]
        chunks.append(chunk)
    return chunks

def chunk_by_paragraph(text, max_size=400):
    paragraphs = [p.strip() for p in text.split("\n") if p.strip()]
    chunks = []
    current = ""
    for para in paragraphs:
        if len(current) + len(para) + 2 <= max_size:
            current = current + "\n" + para if current else para
        else:
            if current:
                chunks.append(current)
            if len(para) > max_size:
                for i in range(0, len(para), max_size - 50):
                    chunks.append(para[i:i + max_size])
            else:
                current = para
    if current:
        chunks.append(current)
    return chunks

chunks = chunk_by_paragraph(text)
print(f"Total chunks: {len(chunks)}")

Total chunks: 269


In [15]:
emb = model.encode(
    chunks,
    convert_to_numpy=True,
    normalize_embeddings=True
)

emb = emb.astype("float32")
index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)

In [16]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

In [17]:
from groq import Groq

client = Groq(api_key=api_key)

In [18]:
import importlib, groq_deepeval
importlib.reload(groq_deepeval)
from groq_deepeval import GroqModel
groq_model = GroqModel(api_key, model="llama-3.1-8b-instant")

In [9]:
query = "What was douchebag referring to in the movie?"
query_embedding = model.encode(
    query,
    convert_to_numpy=True,
    normalize_embeddings=True
)

query_embedding = query_embedding.reshape(1, -1).astype("float32")
retrieved_chunks = []
distances, indices = index.search(query_embedding, 10)
for idx in indices[0]:
    retrieved_chunks.append(chunks[idx])

context = "\n\n".join(retrieved_chunks)

distances, indices = index.search(query_embedding, 10)

In [ ]:
prompt = f"""
You are a question-answering assistant.

Use ONLY the provided context.

Rules:

1. Never use outside knowledge.
2. Never infer information that is not explicitly stated.
3. If the answer is missing, reply exactly:
   "No information available in the provided context."
4. After every answer, include the exact sentence(s) from the context that support your answer.
5. If no supporting sentence exists, return only:
   "No information available in the provided context."

Context:
{context}

Question:
{query}

"""

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)
print("Question:" ,query)
print("Answer:" ,response.choices[0].message.content)

In [19]:
with open("test_dataset.json", "r", encoding="utf-8") as f:
    dataset = json.load(f)

In [20]:
def ask_rag(query):
    query_embedding = model.encode([query]).astype("float32")
    distances, indices = index.search(query_embedding, 20)
    candidate_chunks = [chunks[i] for i in indices[0]]

    pairs = [[query, chunk] for chunk in candidate_chunks]
    scores = reranker.predict(pairs)
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:5]
    retrieved_chunks = [candidate_chunks[i] for i in top_indices]

    context = "\n\n".join(retrieved_chunks)
    prompt = f"""You are a precise question-answering assistant.

Use ONLY the context below to answer. Do not use any outside knowledge.
Be concise and direct. If the answer is not in the context, say exactly:
"No information available in the provided context."

Context:
{context}

Question:
{query}

Answer:"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    answer = response.choices[0].message.content
    return answer, retrieved_chunks

In [22]:
import json
import os

# Load previous results
if os.path.exists("rag_outputs_reranked.json"):
    with open("rag_outputs_reranked.json", "r", encoding="utf-8") as f:
        rag_output = json.load(f)
else:
    rag_output = []

print("Already completed:", len(rag_output))

Already completed: 0


In [23]:
start = len(rag_output)

print("Resuming from question", start + 1)

Resuming from question 1


In [ ]:
for item in dataset[start:]:

    question = item["question"]
    expected_answer = item["answer"]

    print(f"\nRunning: {question}")

    try:
        rag_answer, retrieved_chunks = ask_rag(question)

        rag_output.append({
            "question": question,
            "expected_answer": expected_answer,
            "rag_answer": rag_answer,
            "retrieved_chunks": retrieved_chunks
        })

        # Save immediately after each successful question
        with open("rag_outputs_reranked.json", "w", encoding="utf-8") as f:
            json.dump(rag_output, f, indent=4, ensure_ascii=False)

        print("✓ Saved", len(rag_output), "answers")

    except Exception as e:
        print("Error:", e)
        print("Progress saved. Resume later.")
        break


Running: What is the name of the boy in the story?
✓ Saved 1 answers

Running: Where is Tom from?
✓ Saved 2 answers

Running: What is the name of the girl in the story?
✓ Saved 3 answers

Running: Where is Summer from?
✓ Saved 4 answers

Running: What does Tom believe in?
✓ Saved 5 answers

Running: What is Summer's attitude towards love?
✓ Saved 6 answers

Running: What is Tom's occupation?
✓ Saved 7 answers

Running: What is Summer's job?
✓ Saved 8 answers

Running: How does Tom meet Summer?
✓ Saved 9 answers

Running: What is the date when Tom meets Summer?
✓ Saved 10 answers

Running: What is the name of Tom's half-sister?
✓ Saved 11 answers

Running: Why does Summer want to stop seeing Tom?
✓ Saved 12 answers

Running: What is the name of the song quoted by Summer in her high school yearbook?
✓ Saved 13 answers

Running: What is the name of the album by Belle & Sebastian that experiences a spike in sales in Michigan?
✓ Saved 14 answers

Running: Where does Summer work during her 

In [20]:
print(test_cases[0].__dict__)

IndexError: list index out of range

In [ ]:
import json

from deepeval.metrics import (
    FaithfulnessMetric,
    AnswerRelevancyMetric,
    HallucinationMetric
)
from deepeval.test_case import LLMTestCase

faithfulness = FaithfulnessMetric(model=groq_model)
answer_relevancy = AnswerRelevancyMetric(model=groq_model)
hallucination = HallucinationMetric(model=groq_model)

In [18]:
import json
from deepeval.test_case import LLMTestCase

with open("rag_outputs_partial.json", "r", encoding="utf-8") as f:
    rag_outputs = json.load(f)

test_cases = []

for item in rag_outputs:

    question = item["question"]
    rag_answer = item["rag_answer"]
    expected_answer = item["expected_answer"]
    retrieved_chunks = item["retrieved_chunks"]

    tc = LLMTestCase(
        input=question,
        actual_output=rag_answer,
        expected_output=expected_answer,
        retrieval_context=retrieved_chunks,
        context=retrieved_chunks
    )

    test_cases.append(tc)

print(len(test_cases))

180


In [23]:
import os

EVAL_RESULTS_FILE = "eval_results.json"
small_test = test_cases[:20]

if os.path.exists(EVAL_RESULTS_FILE):
    with open(EVAL_RESULTS_FILE, "r") as f:
        eval_results = json.load(f)
else:
    eval_results = []

done_questions = {r["question"] for r in eval_results}
print(f"Already evaluated: {len(eval_results)}, Remaining: {len(small_test) - len(eval_results)}")

for i, tc in enumerate(small_test):
    if tc.input in done_questions:
        print(f"[{i+1}/{len(small_test)}] Skipping (already done): {tc.input}")
        continue

    print(f"
[{i+1}/{len(small_test)}] Q: {tc.input}")
    try:
        faithfulness.measure(tc)
        answer_relevancy.measure(tc)
        hallucination.measure(tc)

        result = {
            "question": tc.input,
            "faithfulness": faithfulness.score,
            "answer_relevancy": answer_relevancy.score,
            "hallucination": hallucination.score,
        }
        eval_results.append(result)

        with open(EVAL_RESULTS_FILE, "w") as f:
            json.dump(eval_results, f, indent=4)

        print(f"  Faithfulness:     {faithfulness.score:.2f} ({'PASS' if faithfulness.is_successful() else 'FAIL'})")
        print(f"  Answer Relevancy: {answer_relevancy.score:.2f} ({'PASS' if answer_relevancy.is_successful() else 'FAIL'})")
        print(f"  Hallucination:    {hallucination.score:.2f} ({'PASS' if hallucination.is_successful() else 'FAIL'})")
    except Exception as e:
        print(f"  ERROR: {e}")
        print("  Skipping and continuing...")

    time.sleep(10)

print("
" + "="*40)
print("FINAL SUMMARY")
print("="*40)
for metric in ["faithfulness", "answer_relevancy", "hallucination"]:
    vals = [r[metric] for r in eval_results]
    if vals:
        print(f"{metric:20s}: avg={sum(vals)/len(vals):.2f}, min={min(vals):.2f}, max={max(vals):.2f}")


[1/20] Q: What is the name of the boy in the story?


  Faithfulness:     0.83 (PASS)
  Answer Relevancy: 0.67 (PASS)
  Hallucination:    1.00 (FAIL)

[2/20] Q: Where is Tom from?


  Faithfulness:     1.00 (PASS)
  Answer Relevancy: 0.50 (PASS)
  Hallucination:    1.00 (FAIL)

[3/20] Q: What is the name of the girl in the story?


  ERROR: Error code: 400 - {'error': {'message': "Failed to generate JSON. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'json_validate_failed', 'failed_generation': 'max completion tokens reached before generating a valid document'}}

[4/20] Q: Where is Summer from?


  ERROR: Error code: 400 - {'error': {'message': "Failed to generate JSON. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'json_validate_failed', 'failed_generation': 'max completion tokens reached before generating a valid document'}}

[5/20] Q: What does Tom believe in?


  ERROR: Error code: 400 - {'error': {'message': "Failed to generate JSON. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'json_validate_failed', 'failed_generation': 'max completion tokens reached before generating a valid document'}}

[6/20] Q: What is Summer's attitude towards love?


KeyboardInterrupt: 